# **Quantum-Classical Hybrid Machine Learning for Image Classification**

# **Quanvolution + MLP**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import pennylane as qml
from pennylane import numpy as np
import time
from collections import defaultdict

## 1. Quantum Circuit

In [ ]:
dev = qml.device("default.qubit", wires=4)

def circuit_13_layer(weights, wires):
    """
    - RY gates: 4 tham số (0-3)
    - CRZ entanglement: 4 tham số (4-7)
    - RY gates: 4 tham số (8-11)
    - CRZ entanglement: 4 tham số (12-15)
    Total: 16 tham số per layer
    """
    # RY column 1 (parameters 0-3)
    for i in range(4):
        qml.RY(weights[i], wires=wires[i])
    
    # CRZ entanglement 1 (parameters 4-7)
    qml.CRZ(weights[4], wires=[wires[3], wires[0]])
    qml.CRZ(weights[5], wires=[wires[2], wires[3]])
    qml.CRZ(weights[6], wires=[wires[1], wires[2]])
    qml.CRZ(weights[7], wires=[wires[0], wires[1]])
    
    # RY column 2 (parameters 8-11)
    for i in range(4):
        qml.RY(weights[i + 8], wires=wires[i])
    
    # CRZ entanglement 2 (parameters 12-15)
    qml.CRZ(weights[12], wires=[wires[3], wires[2]])
    qml.CRZ(weights[13], wires=[wires[0], wires[3]])
    qml.CRZ(weights[14], wires=[wires[1], wires[0]])
    qml.CRZ(weights[15], wires=[wires[2], wires[1]])

@qml.qnode(dev, interface="torch")
def quantum_filter(inputs, weights):
    """
    4 variables/qubit encoding
    Encode 4x4 = 16 pixels vào 4 qubits
    RZ-RX-RZ-RX encoding
    """
    for i in range(4):
        qml.RZ(inputs[i*4 + 0], wires=i)
        qml.RX(inputs[i*4 + 1], wires=i)
        qml.RZ(inputs[i*4 + 2], wires=i)
        qml.RX(inputs[i*4 + 3], wires=i)
    
    # 3 parametric layers (3 × 16 = 48 parameters total)
    for layer_weights in weights:
        circuit_13_layer(layer_weights, wires=range(4))
        
    # Pauli-Z expectation values (Fig 1, 118, 193)
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

# 2. Hybrid Model Architecture

In [ ]:
class QuanvLayer(nn.Module):
    """Quanvolution Layer - Quantum Feature Extraction"""
    def __init__(self, n_layers=3, trainable=True):
        super().__init__()
        self.n_layers = n_layers
        
        if trainable:
            # Trainable: khởi tạo ngẫu nhiên trong [-pi, pi]
            self.weights = nn.Parameter(torch.randn(n_layers, 16) * np.pi)
        else:
            # Non-trainable: cố định random weights
            self.weights = nn.Parameter(torch.randn(n_layers, 16) * np.pi, requires_grad=False)
        
    def forward(self, x):
        batch, _, h, w = x.shape
        # Kernel 4x4, Stride 4
        out_dim = (h - 4) // 4 + 1
        features = torch.zeros(batch, 4, out_dim, out_dim).to(x.device)
        
        for b in range(batch):
            for i in range(out_dim):
                for j in range(out_dim):
                    # Cắt vùng 4x4, normalize to [0, 2pi]
                    region = x[b, 0, i*4:i*4+4, j*4:j*4+4].reshape(-1) * 2 * np.pi
                    q_out = quantum_filter(region, self.weights)
                    features[b, :, i, j] = torch.stack(q_out)
        return features


class HybridNet(nn.Module):
    """Hybrid Network - Fig 3: Quanv -> Flatten -> Linear -> ReLU -> Linear -> Softmax"""
    def __init__(self, n_layers=3, trainable=True, num_classes=3):
        super().__init__()
        self.quanv = QuanvLayer(n_layers=n_layers, trainable=trainable)
        
        # Input: 4 channels × 3×3 feature maps (với 14x14 input + stride 4)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4 * 3 * 3, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
            # KHÔNG dùng Softmax vì CrossEntropyLoss đã có sẵn
        )
    
    def forward(self, x):
        return self.mlp(self.quanv(x))

# 3. Dataset 

In [ ]:
# 6 subsets theo Table I
all_subsets = {
    "MNIST_179": [1, 7, 9],
    "MNIST_246": [2, 4, 6],
    "MNIST_358": [3, 5, 8],
    "Fashion_012": [0, 1, 2],
    "Fashion_345": [3, 4, 5],
    "Fashion_678": [6, 7, 8]
}

def get_subset_loaders(name, classes, samples_per_class=400):
    """
    - Giảm 28x28 -> 14x14 bằng MaxPool2d
    - Lấy ~400 mẫu mỗi lớp = 1200 tổng
    - Chia 600 train / 600 validation
    """
    # Transform: ToTensor -> MaxPool2d(2) để giảm 28x28 -> 14x14
    transform = transforms.Compose([
        transforms.ToTensor(),
        nn.MaxPool2d(2),  # MaxPool2d thay vì Resize - đúng theo paper
    ])
    
    ds_class = datasets.MNIST if "MNIST" in name else datasets.FashionMNIST
    full_ds = ds_class(root='./data', train=True, download=True, transform=transform)
    
    # Lấy đúng ~400 mẫu mỗi lớp (không phải first 1200)
    class_indices = defaultdict(list)
    for i, (_, label) in enumerate(full_ds):
        if label in classes and len(class_indices[label]) < samples_per_class:
            class_indices[label].append(i)
    
    # Ghép lại thành 1 list
    indices = []
    for c in classes:
        indices.extend(class_indices[c])
    
    # Shuffle indices
    np.random.seed(42)
    np.random.shuffle(indices)
    
    subset = Subset(full_ds, indices)
    
    # Chia 600 train / 600 validation
    train_loader = DataLoader(Subset(subset, range(600)), batch_size=4, shuffle=True)
    val_loader = DataLoader(Subset(subset, range(600, 1200)), batch_size=4)
    
    return train_loader, val_loader

# 4. Training

In [ ]:
def train_and_evaluate(train_loader, val_loader, model, trainable=True, num_epochs=10):
    """
    - Optimizer: Adagrad
    - Learning rate: 0.5
    - Loss: CrossEntropyLoss
    """
    optimizer = optim.Adagrad(model.parameters(), lr=0.5)
    criterion = nn.CrossEntropyLoss()
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss, train_correct = 0, 0
        
        for data, target in train_loader:
            target_mapped = torch.tensor([0, 1, 2])[torch.tensor([c == target[i] for i, c in enumerate([0,1,2]) for _ in range(len(target))]).reshape(3, len(target)).any(0).int()].long() if len(target) > 0 else target

            target_idx = torch.zeros(len(target), dtype=torch.long)
            for i, t in enumerate(target):
                for j, c in enumerate([0,1,2]):
                    if t.item() == c:
                        target_idx[i] = j
                        break
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target_idx)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * data.size(0)
            train_correct += (output.argmax(1) == target_idx).sum().item()
        
        # Validation
        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for data, target in val_loader:
                target_idx = torch.zeros(len(target), dtype=torch.long)
                for i, t in enumerate(target):
                    for j, c in enumerate([0,1,2]):
                        if t.item() == c:
                            target_idx[i] = j
                            break
                output = model(data)
                loss = criterion(output, target_idx)
                val_loss += loss.item() * data.size(0)
                val_correct += (output.argmax(1) == target_idx).sum().item()
        
        train_loss_epoch = train_loss / 600
        val_loss_epoch = val_loss / 600
        train_acc = train_correct / 600
        val_acc = val_correct / 600
        print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss_epoch:.4f} | Val Loss: {val_loss_epoch:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    
    elapsed = time.time() - start_time
    return train_acc, val_acc, elapsed

# 5. Main Training Loop - So sánh Trainable vs Non-trainable Filters

In [6]:
# Chỉ chạy 1 subset để test nhanh (MNIST_179)
subset_name = "MNIST_179"
classes = [1, 7, 9]

print(f"{'='*50}")
print(f"DATASET: {subset_name} - Classes: {classes}")
print(f"{'='*50}")

# Load data
train_loader, val_loader = get_subset_loaders(subset_name, classes)

# ======== NON-TRAINABLE FILTERS ========
print(f"\n{'='*20} NON-TRAINABLE FILTERS {'='*20}")
model_non_trainable = HybridNet(n_layers=3, trainable=False, num_classes=3)
train_acc_nt, val_acc_nt, time_nt = train_and_evaluate(
    train_loader, val_loader, model_non_trainable, 
    trainable=False, num_epochs=10
)
print(f"Time: {time_nt:.2f}s")

# ======== TRAINABLE FILTERS ========
print(f"\n{'='*20} TRAINABLE FILTERS {'='*20}")
model_trainable = HybridNet(n_layers=3, trainable=True, num_classes=3)
train_acc_t, val_acc_t, time_t = train_and_evaluate(
    train_loader, val_loader, model_trainable, 
    trainable=True, num_epochs=10
)
print(f"Time: {time_t:.2f}s")

# ======== SUMMARY ========
print(f"\n{'='*50}")
print(f"SUMMARY - {subset_name}")
print(f"{'='*50}")
print(f"                     | Non-trainable | Trainable")
print(f"Train Accuracy       | {train_acc_nt:.4f}     | {train_acc_t:.4f}")
print(f"Validation Accuracy | {val_acc_nt:.4f}     | {val_acc_t:.4f}")
print(f"Training Time (s)   | {time_nt:.2f}       | {time_t:.2f}")

DATASET: MNIST_179 - Classes: [1, 7, 9]

==================== NON-TRAINABLE FILTERS ====================
Epoch  1 | Train Loss: 0.7561 | Val Loss: 0.5727 | Train Acc: 0.6950 | Val Acc: 0.6950
Epoch  2 | Train Loss: 0.4489 | Val Loss: 0.4324 | Train Acc: 0.7983 | Val Acc: 0.7983
Epoch  3 | Train Loss: 0.3925 | Val Loss: 0.4057 | Train Acc: 0.8267 | Val Acc: 0.8217
Epoch  4 | Train Loss: 0.3547 | Val Loss: 0.3739 | Train Acc: 0.8333 | Val Acc: 0.8250
Epoch  5 | Train Loss: 0.3118 | Val Loss: 0.3627 | Train Acc: 0.8667 | Val Acc: 0.8250
Epoch  6 | Train Loss: 0.2794 | Val Loss: 0.3520 | Train Acc: 0.8617 | Val Acc: 0.8550
Epoch  7 | Train Loss: 0.2507 | Val Loss: 0.3890 | Train Acc: 0.8833 | Val Acc: 0.8367
Epoch  8 | Train Loss: 0.2395 | Val Loss: 0.3626 | Train Acc: 0.8950 | Val Acc: 0.8483
Epoch  9 | Train Loss: 0.2030 | Val Loss: 0.3703 | Train Acc: 0.9217 | Val Acc: 0.8450
Epoch 10 | Train Loss: 0.2052 | Val Loss: 0.3914 | Train Acc: 0.9083 | Val Acc: 0.8317
Time: 452.80s

==========

# 6. So sánh với Table I trong Paper

Theo Table I (paper), kết quả MNIST_179 sau 10 epochs:

| Method | Train Loss | Val Loss | Train Acc | Val Acc |
|--------|------------|----------|-----------|----------|
| Non-trainable | 0.3717 | 0.5537 | 0.8416 | 0.7830 |
| Trainable | 0.3881 | 0.5338 | 0.8500 | 0.7733 |

In [10]:
print("COMPARISON WITH PAPER (Table I - MNIST_179)")

print(f"                                             | Paper    | Ours                ")
print(f"Non-trainable - Train Accuracy               | 0.8416   | {train_acc_nt:.4f}  ")
print(f"Non-trainable - Validation Accuracy          | 0.7830   | {val_acc_nt:.4f}    ")
print(f"Trainable - Train Accuracy                   | 0.8500   | {train_acc_t:.4f}   ")
print(f"Trainable - Validation Accuracy              | 0.7733   | {val_acc_t:.4f}     ")

COMPARISON WITH PAPER (Table I - MNIST_179)
                                             | Paper    | Ours                
Non-trainable - Train Accuracy               | 0.8416   | 0.9083  
Non-trainable - Validation Accuracy          | 0.7830   | 0.8317    
Trainable - Train Accuracy                   | 0.8500   | 0.9383   
Trainable - Validation Accuracy              | 0.7733   | 0.8650     
